<p align="center"><img src="https://raw.githubusercontent.com/arab-future-academy/arab-future-academy/main/logo.png" width="200"></p>

# 🚀 ComfyUI on Lightning.ai — Arabic Future Academy

**هذا النوتبوك مصمم خصيصاً لـ Lightning.ai Studio** — بنفس الكتالوج والحزم الموجودة في نسخة Colab.

### الفروقات عن Colab:
| | Colab | Lightning.ai |
|---|---|---|
| التخزين | مؤقت (`/content`) | **دائم** (Studio disk) |
| GPU | T4 مجاني (حد استخدام) | T4/A10G (~22 ساعة شهرياً) |
| الوصول | Cloudflared tunnel | **Port Viewer** (مدمج) |
| PyTorch | قد تحتاج تثبيت | **مثبّت مسبقاً** |
| Google Drive | مطلوب | غير مطلوب |

### خطوات البدء:
1. أنشئ Studio جديد من <https://lightning.ai/comfy>
2. حوّل لـ **GPU** (T4)
3. شغّل هذه الخلايا بالترتيب

---


# **الإعدادات المركزية**

**نفس إعدادات Colab — نفس الكتالوج المثبّت والمراجع (SHA).**

اختر حزمة `BUNDLE` واحدة في الجلسة (أو استخدم `none` + `EXTRA_FILES`).

**ملاحظة Lightning.ai:** التخزين دائم — الموديلات تبقى بين الجلسات! لا حاجة لإعادة التحميل.


In [ ]:
#@title 0. Central Configuration
#@markdown Same catalog and bundles as the Colab version.

# --- Model selection ---
#@markdown **Model bundle:** curated sets from the pinned manifest.
BUNDLE = "none" #@param ["none", "z-image-turbo-t4", "z-image-turbo", "sd15-classic", "qwen-edit-2511", "qwen-image-2.1-edit", "flux2-klein", "chatterbox-tts", "depth-3d-trellis", "3d-pixal3d-trellis2", "wan-vace-t4", "wan-scail2-t4", "extras", "llm-qwen3-8b-t4", "llm-qwen2.5-7b-t4", "llm-qwen2.5-3b-t4", "llm-qwen2.5-1.5b-t4", "llm_gemma4_text_gen", "llm_gemma4_text_gen_t4", "llm_qwen3_5_text_gen", "llm_qwen3_5_text_gen_t4", "starter-t4", "wan-scail2-full", "wan-vace-full"]
#@markdown **Extra files:** one per line — manifest dest or hf://repo/path/file
EXTRA_FILES = "" #@param {type:"string"}

# --- Paths (Lightning.ai persistent disk) ---
import os, tempfile

# Diagnose available paths
print("[PATH] Diagnosing storage...")
for p in ["/mnt/user-data", "/mnt", "/workspace", "/tmp", os.path.expanduser("~")]:
    exists = os.path.isdir(p)
    writable = os.access(p, os.W_OK) if exists else False
    print("  {:30} exists={} writable={}".format(p, exists, writable))

def _find_writable():
    """Find a writable directory for ComfyUI. Prefer persistent storage."""
    candidates = [
        ("/mnt/user-data", "ComfyUI"),           # Lightning.ai persistent
        ("/mnt", "user-data/ComfyUI"),            # fallback
        ("/workspace", "ComfyUI"),                # might be writable
        (os.path.expanduser("~"), "ComfyUI"),     # home
        (tempfile.gettempdir(), "ComfyUI"),       # /tmp (ephemeral)
    ]
    for base, sub in candidates:
        full = os.path.join(base, sub) if sub else base
        # Check if base exists and is writable (or can create full dir)
        check = base if os.path.isdir(base) else os.path.dirname(full)
        if os.path.isdir(check) and os.access(check, os.W_OK):
            return full
    return os.path.join(tempfile.gettempdir(), "ComfyUI")

LOCAL_WORKSPACE = _find_writable()
os.makedirs(LOCAL_WORKSPACE, exist_ok=True)
print("[PATH] Using: {}".format(LOCAL_WORKSPACE))
print("[PATH] Persistent: {}".format("YES — models survive restarts" if "/mnt/user-data" in LOCAL_WORKSPACE else "NO — models lost on restart!"))
if "/mnt/user-data" not in LOCAL_WORKSPACE:
    print("[PATH] WARNING: /mnt/user-data not available. Check Studio Environment.")

MANIFEST_URL = "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml"
HF_XET_HIGH_PERF = False

# --- Info ---
_n_extra = len([l for l in EXTRA_FILES.splitlines() if l.strip() and not l.strip().startswith("#")])
print("Configuration loaded.")
print("  Bundle: {} · Extra files: {}".format(BUNDLE, _n_extra))
print("  Workspace: {}".format(LOCAL_WORKSPACE))
print("  Persistent: {}".format("Yes (/mnt/user-data)" if "/mnt/user-data" in LOCAL_WORKSPACE else "No"))


# **تهيئة النظام**

**في Lightning.ai: PyTorch مثبّت مسبقاً — نحتاج فقط تثبيت ComfyUI والمكتبات.**

### قبل التشغيل:
- تأكد إن الـ **Studio** يعمل على **GPU** (T4)
- من الأعلى: **Studio Environment → Switch to GPU**

### ماذا يحدث:
- فحص GPU + PyTorch
- تثبيت/تحديث ComfyUI
- تثبيت المكتبات الأساسية


In [ ]:
#@title 1. System Initialization
#@markdown Installs ComfyUI and core dependencies. PyTorch is pre-installed on Lightning.ai.

import os, subprocess, sys, shutil

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
print("[SYSTEM] Workspace: {}".format(WORKSPACE))
print("[ENV] Python: {}".format(sys.executable))
print("[ENV] Conda env: {}".format(os.environ.get("CONDA_DEFAULT_ENV", "none")))

# Install torch FIRST — ComfyUI needs it to boot
print("\n[TORCH] Ensuring PyTorch is installed...")
try:
    import torch
    print("[TORCH] Found: {} · CUDA: {}".format(torch.__version__, torch.cuda.is_available()))
except ImportError:
    print("[TORCH] Not found — installing CUDA torch...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu124"], check=False)
    try:
        import importlib
        import torch
        importlib.reload(torch)
        print("[TORCH] Installed: {} · CUDA: {}".format(torch.__version__, torch.cuda.is_available()))
    except Exception as e:
        print("[TORCH] Install/reload failed: {}".format(e))

def run(cmd, cwd=None, check=True):
    print("$ {}".format(" ".join(cmd) if isinstance(cmd, list) else cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True)
    if check and r.returncode != 0:
        print("[WARN] Command returned {}".format(r.returncode))

# 1. GPU check + torch install in ACTIVE environment
print("\n[GPU] Checking...")
print("[ENV] Python: {}".format(sys.executable))
print("[ENV] Conda env: {}".format(os.environ.get("CONDA_DEFAULT_ENV", "none")))

# Ensure pip exists in this env
try:
    import torch
    _have_torch = True
    print("[TORCH] Found: {}".format(torch.__version__))
except ImportError:
    _have_torch = False
    print("[TORCH] Not installed in this environment.")

# Check physical GPU
_has_gpu = False
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True, timeout=5)
    if r.returncode == 0 and r.stdout.strip():
        _has_gpu = True
        print("[GPU] Physical GPU: {}".format(r.stdout.strip()))
except Exception:
    pass

if _has_gpu and not _have_torch:
    print("[TORCH] GPU detected but torch missing — installing CUDA torch...")
    run([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cu124"])
    try:
        import importlib
        import torch
        importlib.reload(torch)
        print("[TORCH] Installed: {}".format(torch.__version__))
        print("[TORCH] CUDA available: {}".format(torch.cuda.is_available()))
        if torch.cuda.is_available():
            print("[TORCH] GPU: {}".format(torch.cuda.get_device_name(0)))
    except Exception as e:
        print("[TORCH] Reload failed: {} — restart Studio".format(e))
elif _has_gpu and _have_torch:
    if torch.cuda.is_available():
        print("[OK] GPU: {} · torch {}".format(torch.cuda.get_device_name(0), torch.__version__))
        print("[OK] VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
    else:
        print("[WARN] GPU exists but torch.cuda=False — reinstalling...")
        run([sys.executable, "-m", "pip", "install", "--force-reinstall", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu124"])
        print("[ACTION] Restart Studio, then re-run all cells.")
        raise SystemExit("\n[PAUSED] Restart Studio after torch install.")
elif not _has_gpu:
    print("[WARN] No GPU detected. Switch to GPU: Studio Environment > Switch to GPU")
    print("[WARN] Continuing with CPU — models will be very slow.")

# 2. Disk check
for p in ["/mnt/user-data", "/workspace", "/"]:
    if os.path.exists(p):
        total, used, free = shutil.disk_usage(p)
        print("[OK] Disk free ({}) : {:.1f} GB".format(p, free / (1024**3)))
        break

# 3. Clone / update ComfyUI
if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    print("\n[COMFYUI] Updating...")
    run(["git", "pull"], cwd=WORKSPACE)
else:
    print("\n[COMFYUI] Cloning into {}...".format(WORKSPACE))
    os.makedirs(WORKSPACE, exist_ok=True)
    # git clone fails if dir is non-empty — clone to temp then move
    import glob as _glob
    existing = _glob.glob(os.path.join(WORKSPACE, "*"))
    if existing:
        print("[COMFYUI] Dir not empty ({} files) — cloning to temp...".format(len(existing)))
        tmp_clone = WORKSPACE + "_clone_tmp"
        run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", tmp_clone])
        for item in _glob.glob(os.path.join(tmp_clone, "*")):
            shutil.move(item, WORKSPACE)
        shutil.rmtree(tmp_clone, ignore_errors=True)
    else:
        run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", WORKSPACE])

# 4. Manager
manager_path = os.path.join(WORKSPACE, "custom_nodes", "ComfyUI-Manager")
if not os.path.exists(manager_path):
    print("[MANAGER] Installing...")
    os.makedirs(os.path.join(WORKSPACE, "custom_nodes"), exist_ok=True)
    run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager.git", manager_path])
else:
    print("[MANAGER] Verified.")

# 5. Dependencies
print("\n[PYTHON] Installing requirements...")
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORKSPACE)
run([sys.executable, "-m", "pip", "install", "-q", "gguf", "piexif", "huggingface_hub", "pyyaml", "requests"])

# 5b. Also ensure torch is visible to ComfyUI-Manager (checks sys.modules)
print("\n[PYTHON] Verifying torch is importable...")
try:
    import torch
    print("[PYTHON] torch {} · CUDA {}".format(torch.__version__, torch.cuda.is_available()))
except ImportError:
    print("[PYTHON] torch NOT importable — installing...")
    run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cu124"])

# 6. Xet off
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = os.path.join(WORKSPACE, ".hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("\n[SYSTEM READY] Initialization complete.")


# **تسجيل الدخول لهافينج فيس**

**مطلوب فقط إذا كنت تحمّل من مستودعات خاصة أو تريد rate limit أعلى.**

احفظ التوكن في: **Studio Settings → Secrets → HF_TOKEN**

بدون توكن: التحميل يعمل لكن بسرعة أقل (rate limit مجهول).


In [ ]:
#@title 1b. Hugging Face Login
#@markdown Required for uploading to awAyham/comfyui-models.
import os

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    from huggingface_hub import login, HfApi
    login(token=hf_token)
    api = HfApi()
    
    # Check repo access
    repo = "awAyham/comfyui-models"
    try:
        info = api.model_info(repo)
        print("[HF] Logged in.")
        print("[HF] Repo: {}".format(repo))
        # List existing files
        files = api.list_repo_files(repo)
        print("[HF] Existing files ({}):".format(len(files)))
        for f in files[:10]:
            print("  {}".format(f))
        if len(files) > 10:
            print("  ... and {} more".format(len(files) - 10))
    except Exception as e:
        print("[HF] Logged in, but repo check failed: {}".format(e))
        print("[HF] Repo will be created on first upload.")
else:
    print("[HF] No HF_TOKEN found!")
    print("[HF] To upload models, add HF_TOKEN:")
    print("     Studio Settings > Secrets > HF_TOKEN")
    print("[HF] Get token: https://huggingface.co/settings/tokens")


# **تحميل الموديلات**

**نفس كتالوج Colab — تحميل مباشر من Hugging Face بمراجع مثبّتة.**

### ⚡ ميزة Lightning.ai:
الملفات تُحفظ في القرص **الدائم** — لا تُحذف بين الجلسات!

اختر حزمة في `BUNDLE` (خلية الإعدادات) ثم شغّل.


In [ ]:
#@title 2. Model Pull (manifest-driven)
#@markdown Pulls the selected bundle from pinned upstream sources.

import os, sys, shutil, tempfile
import requests, yaml
from huggingface_hub import hf_hub_download, HfApi

BUNDLE = globals().get("BUNDLE", "none")
EXTRA_FILES = globals().get("EXTRA_FILES", "")
MANIFEST_URL = globals().get("MANIFEST_URL", "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml")
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
MIN_FREE_DISK_GB = 15

# Disable Xet
os.environ["HF_HUB_DISABLE_XET"] = "1"

MODELS_ROOT = os.path.join(LOCAL_WORKSPACE, "models")

# 1) Fetch manifest
print("[MANIFEST] Fetching {}".format(MANIFEST_URL))
head = requests.head(MANIFEST_URL, timeout=20, allow_redirects=True)
head.raise_for_status()
manifest = yaml.safe_load(requests.get(MANIFEST_URL, timeout=30).text)
FILES_BY_DEST = {f["dest"]: f for f in manifest.get("files", [])}
BUNDLES = manifest.get("bundles", {})
print("[MANIFEST] {} files · bundles: {}".format(len(FILES_BY_DEST), ", ".join(BUNDLES)))

# 2) Resolve selection
selected = []
if BUNDLE and BUNDLE != "none":
    if BUNDLE not in BUNDLES:
        raise SystemExit("[PULL] Unknown bundle '{}'. Available: {}".format(BUNDLE, list(BUNDLES)))
    for dest in BUNDLES[BUNDLE]:
        if dest not in FILES_BY_DEST:
            raise SystemExit("[PULL] Bundle '{}' references missing entry: {}".format(BUNDLE, dest))
        selected.append(dict(FILES_BY_DEST[dest]))

for raw in (EXTRA_FILES or "").replace(",", "\n").split("\n"):
    spec = raw.strip()
    if not spec or spec.startswith("#"):
        continue
    if spec in FILES_BY_DEST:
        selected.append(dict(FILES_BY_DEST[spec]))
    elif spec.startswith("hf://"):
        ref = spec[len("hf://"):]
        repo_id, _, remote = ref.partition("/")
        if not remote:
            raise SystemExit("[PULL] Bad hf:// ref: {}".format(spec))
        sha = HfApi().model_info(repo_id).sha
        selected.append({"dest": "extras/" + os.path.basename(remote), "source": "hf://" + repo_id,
                         "remote": remote, "revision": sha, "size_gb": None})
    else:
        raise SystemExit("[PULL] Unrecognized: {}".format(spec))

# de-dupe
deduped, seen = [], set()
for j in selected:
    if j["dest"] in seen:
        continue
    seen.add(j["dest"])
    deduped.append(j)
selected = deduped

if not selected:
    print("[PULL] Nothing selected. Skipping.")
else:
    need_gb = sum(j["size_gb"] for j in selected if j.get("size_gb"))
    _check_dir = LOCAL_WORKSPACE if os.path.isdir(LOCAL_WORKSPACE) else os.path.dirname(LOCAL_WORKSPACE)
    if not os.path.isdir(_check_dir):
        _check_dir = "/"
    free_gb = shutil.disk_usage(_check_dir).free / (1024**3)
    print("[DISK] Need ~{:.1f} GB · free: {:.1f} GB".format(need_gb, free_gb))
    if free_gb - need_gb < MIN_FREE_DISK_GB:
        raise SystemExit("[DISK] Not enough space: need {:.1f} + {} margin, have {:.1f} GB".format(need_gb, MIN_FREE_DISK_GB, free_gb))

    results = []
    for j in selected:
        src, dest = j["source"], j["dest"]
        final_path = os.path.join(MODELS_ROOT, dest)
        if os.path.exists(final_path):
            size = os.path.getsize(final_path)
            expect = j.get("size_gb")
            if expect is None or abs(size - expect * 1e9) / max(size, 1) <= 0.02:
                print("   [SKIP] {} ({:.2f} GB)".format(dest, size / 1e9))
                results.append((dest, "skip", size))
                continue
        if src.startswith(("hf://", "archive://")):
            repo_id = src.split("://", 1)[1]
            remote = j.get("remote", dest)
            rev = j.get("revision") or None
            tag = "@" + rev[:10] if rev else "@main"
            print("   [HF] {} :: {} {}".format(repo_id, remote, tag))
            stage = os.path.join(LOCAL_WORKSPACE, ".pull_tmp")
            os.makedirs(stage, exist_ok=True)
            with tempfile.TemporaryDirectory(dir=stage) as td:
                p = hf_hub_download(repo_id=repo_id, filename=remote, revision=rev,
                                    local_dir=td, repo_type="model")
                os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
                shutil.move(p, final_path)
            size = os.path.getsize(final_path)
            print("       -> {} ({:.2f} GB)".format(dest, size / 1e9))
            results.append((dest, "ok", size))
        elif src.startswith(("url://", "http://", "https://")):
            url = src[len("url://"):] if src.startswith("url://") else src
            print("   [URL] {}".format(url))
            os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(final_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
            size = os.path.getsize(final_path)
            results.append((dest, "ok", size))
        else:
            print("   [ERR] unknown source: {}".format(src))
            results.append((dest, "err", 0))

    print("\n[PULL REPORT]")
    ok = sum(1 for _, s_, _ in results if s_ in ("ok", "skip"))
    bad = sum(1 for _, s_, _ in results if s_ == "err")
    for d, s_, sz in results:
        print("   {:4}  {:6.2f} GB  {}".format(s_.upper(), sz / 1e9, d))
    print("[PULL] done: {} ok/skipped · {} failed".format(ok, bad))


# **تشغيل ComfyUI**

**في Lightning.ai لا نحتاج cloudflared — نستخدم Port Viewer المدمج.**

### بعد التشغيل:
1. اضغط **+** أعلى اليمين
2. اختر **Web Apps → Port Viewer**
3. أضف port **8188**
4. افتح الرابط

أو استخدم الرابط المباشر: `http://localhost:8188` داخل الـ Studio.


In [ ]:
#@title 3. Start ComfyUI
#@markdown Boots ComfyUI. Use Port Viewer (port 8188) to access the UI.

import subprocess, time, socket, os, sys

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
PORT = 8188
LOG = os.path.join(WORKSPACE, "comfyui_boot.log")

def port_open(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        return s.connect_ex(("127.0.0.1", port)) == 0
    finally:
        s.close()

# GPU check
try:
    import torch
    if not torch.cuda.is_available():
        print("[WARN] No CUDA — ComfyUI will use CPU (very slow).")
        print("[WARN] Switch to GPU: Studio Environment > Switch to GPU")
except ImportError:
    print("[WARN] torch missing — re-run cell 1.")

# Boot
if port_open(PORT):
    print("[SYSTEM] ComfyUI already on :{}".format(PORT))
elif not os.path.exists(os.path.join(WORKSPACE, "main.py")):
    print("[FATAL] ComfyUI not found at {}".format(WORKSPACE))
    print("        Run cell 1 (System Initialization) first.")
    raise SystemExit(1)
else:
    os.chdir(WORKSPACE)
    open(LOG, "w").close()
    print("[SYSTEM] Booting ComfyUI...")
    cmd = [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT)]
    proc = subprocess.Popen(cmd, stdout=open(LOG, "wb"), stderr=subprocess.STDOUT)
    print("[SYSTEM] PID={} · log={}".format(proc.pid, LOG))

    pos = 0
    up = False
    for i in range(120):
        time.sleep(1)
        if os.path.exists(LOG):
            with open(LOG, "r", errors="replace") as f:
                f.seek(pos)
                chunk = f.read()
                pos = f.tell()
            if chunk:
                sys.stdout.write(chunk)
                sys.stdout.flush()
        if port_open(PORT):
            up = True
            print("\n[OK] Server UP on :{} ({}s)".format(PORT, i + 1))
            break
        if proc.poll() is not None:
            print("\n[FATAL] Process exited ({})".format(proc.returncode))
            print(open(LOG, errors="replace").read()[-2000:])
            raise SystemExit(1)

    if not up:
        print("\n[FATAL] :{} did not open in 120s".format(PORT))
        print(open(LOG, errors="replace").read()[-2000:])
        raise SystemExit(1)

print("=" * 50)
print("  ComfyUI is RUNNING on port {}".format(PORT))
print("")
print("  To access the UI:")
print("  1. Click + (top right)")
print("  2. Web Apps -> Port Viewer")
print("  3. New Port -> 8188")
print("  4. Open the URL")
print("=" * 50)


# **رفع الموديلات إلى HF Repo**

**ارفع الموديلات المحمّلة إلى مستودعك الخاص `awAyham/comfyui-models`.**

### المتطلبات:
- `HF_TOKEN` مُضاف في Secrets
- الموديلات محمّلة (خلية 2)

### كيف يعمل:
1. يقرأ الحزمة المختارة من `BUNDLE`
2. يرفع كل ملف من القرص إلى HF repo
3. يعرض تقرير النجاح/الفشل


In [ ]:
#@title 5. Upload to HF Repo
#@markdown Upload downloaded models to awAyham/comfyui-models.
import os, sys

HF_REPO = "awAyham/comfyui-models"
WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
MODELS_ROOT = os.path.join(WORKSPACE, "models")

hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("[ERROR] No HF_TOKEN — cannot upload.")
    print("        Add: Studio Settings > Secrets > HF_TOKEN")
else:
    from huggingface_hub import HfApi, login
    login(token=hf_token)
    api = HfApi()
    
    # Ensure repo exists
    try:
        api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
        print("[REPO] {} ready".format(HF_REPO))
    except Exception as e:
        print("[REPO] Error: {}".format(e))
    
    # Get bundle selection
    BUNDLE = globals().get("BUNDLE", "none")
    if BUNDLE == "none":
        print("[PUSH] No bundle selected. Set BUNDLE in cell 0 first.")
        print("[PUSH] Available bundles: see cell 0 dropdown")
    else:
        # Fetch manifest to get file list
        import requests, yaml
        MANIFEST_URL = globals().get("MANIFEST_URL", "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml")
        manifest = yaml.safe_load(requests.get(MANIFEST_URL, timeout=30).text)
        
        if BUNDLE not in manifest.get("bundles", {}):
            print("[ERROR] Bundle not found: {}".format(BUNDLE))
        else:
            files = manifest["bundles"][BUNDLE]
            
            print("\n[PUSH] Uploading {} files from bundle '{}'...".format(len(files), BUNDLE))
            print("[PUSH] Target: {}\n".format(HF_REPO))
            
            success = 0
            failed = 0
            skipped = 0
            
            for dest in files:
                local_path = os.path.join(MODELS_ROOT, dest)
                
                if not os.path.exists(local_path):
                    print("  [SKIP] Not downloaded: {}".format(dest))
                    skipped += 1
                    continue
                
                size_gb = os.path.getsize(local_path) / 1e9
                print("  [UPLOAD] {} ({:.2f} GB)...".format(dest, size_gb))
                
                try:
                    api.upload_file(
                        path_or_fileobj=local_path,
                        path_in_repo=dest,
                        repo_id=HF_REPO,
                        repo_type="model",
                    )
                    print("  [OK] {}".format(dest))
                    success += 1
                except Exception as e:
                    print("  [ERROR] {}: {}".format(dest, e))
                    failed += 1
            
            print("\n[PUSH DONE] {} uploaded · {} failed · {} skipped".format(success, failed, skipped))
            print("[PUSH] View: https://huggingface.co/{}".format(HF_REPO))


# **تثبيت Auto-Start (اختياري)**

**يضيف ComfyUI إلى `on_start.sh` ليبدأ تلقائياً مع كل Studio.**


In [ ]:
#@title 4. Auto-Start Setup (optional)
#@markdown Adds ComfyUI boot to on_start.sh for automatic startup.

import os

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")

# Find writable on_start location
candidates = [
    os.path.expanduser("~/.on_start.sh"),
    os.path.join(WORKSPACE, ".on_start.sh"),
    os.path.join(os.environ.get("HOME", "/tmp"), ".on_start.sh"),
]
ON_START = None
for c in candidates:
    parent = os.path.dirname(c)
    if os.path.isdir(parent) and os.access(parent, os.W_OK):
        ON_START = c
        break

if ON_START is None:
    print("[AUTO-START] No writable location found — skipping.")
else:
    boot_line = "cd {} && nohup python main.py --listen 127.0.0.1 --port 8188 > comfyui_boot.log 2>&1 &".format(WORKSPACE)

    if os.path.exists(ON_START):
        content = open(ON_START).read()
        if "main.py" in content:
            print("[AUTO-START] Already configured at {}".format(ON_START))
        else:
            with open(ON_START, "a") as f:
                f.write("\n# ComfyUI auto-start\n" + boot_line + "\n")
            print("[AUTO-START] Added to {}".format(ON_START))
    else:
        with open(ON_START, "w") as f:
            f.write("#!/bin/bash\n# ComfyUI auto-start\n" + boot_line + "\n")
        os.chmod(ON_START, 0o755)
        print("[AUTO-START] Created {}".format(ON_START))

    print("[AUTO-START] Next Studio boot will auto-start ComfyUI.")
